# Kafka Consumer

In [5]:
!pip cache purge --quiet

In [6]:
!pip install confluent-kafka==2.11.0 --quiet

In [7]:
import json
import pandas as pd

from confluent_kafka import Consumer, KafkaException
from IPython.display import clear_output

In [8]:
# Kafka config
conf = {
    "bootstrap.servers": "pkc-921jm.us-east-2.aws.confluent.cloud:9092",
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": "5KM26QOVEDLSBGMB",
    "sasl.password": "cfltG/SHOxHSYCv7HBID0vu1sTNqyuBTuQzrvAv78ewjlwNuWSeLyPPaP5giKrkA",
    "group.id": "jupyter-tick-consumer",
    "auto.offset.reset": "earliest",
    "log_level": 0
}

topic = "tick-data"

In [10]:
def consume_events(consumer, num_messages = -1, max_records_to_show = 20, poll_timeout = 1.0):
    """
    Consume messages from Kafka and display in Jupyter.
    
    consumer            : a configured confluent_kafka.Consumer
    num_messages        : number of messages to consume (-1 for infinite)
    max_records_to_show : max rows to display in notebook
    poll_timeout        : time (sec) to wait for messages per poll
    """
    records = []
    counter = 0

    try:
        while True:
            msg = consumer.poll(timeout=poll_timeout)

            # No message received
            if msg is None:
                if num_messages != -1:
                    break
                else:
                    continue

            # Handle error
            if msg.error():
                raise KafkaException(msg.error())

            # Parse JSON value
            record = json.loads(msg.value().decode("utf-8"))
            records.append(record)
            counter += 1

            # Keep only the latest N records for display
            if len(records) > max_records_to_show:
                records = records[-max_records_to_show:]

            # Display in Jupyter
            clear_output(wait = True)
            df = pd.DataFrame(records)
            display(df)

            # Stop if finite number reached
            if num_messages != -1 and counter >= num_messages:
                break

    except KeyboardInterrupt:
        print("Consumer stopped by user.")
    finally:
        consumer.close()
        print(f"Consumer closed. Total messages consumed: {counter}")

In [11]:
consumer = Consumer(conf)
consumer.subscribe([topic])
consume_events(consumer, num_messages = -1)

,price,symbol,ts
0,142.63,TEST15-FX,2025-08-17 13:59:44
1,96.40,TEST10-FX,2025-08-19 15:11:27
2,64.92,TEST16-FX,2025-08-13 13:27:33
3,215.76,TEST19-FX,2025-08-11 11:03:47
4,89.54,TEST03-FX,2025-08-10 10:45:19
5,52.13,TEST07-FX,2025-08-12 14:22:08
6,74.85,TEST18-FX,2025-08-18 11:36:02
7,381.77,TEST06-FX,2025-08-19 16:22:38
8,33.48,TEST04-FX,2025-08-16 14:55:20
9,118.39,TEST20-FX,2025-08-14 10:34:25


Consumer stopped by user.
Consumer closed. Total messages consumed: 20
